<a href="https://colab.research.google.com/github/Muen1/multilingual-health-qa-africa/blob/main/notebook/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CELL 0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR = '/content/drive/MyDrive/multilingual_health_qa'
DATA_DIR = f'{BASE_DIR}/data'
OUT_DIR  = f'{BASE_DIR}/outputs'
PLOT_DIR = f'{BASE_DIR}/plots'

for d in [OUT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print("Data files:", os.listdir(DATA_DIR))

Mounted at /content/drive
Drive mounted.
Data files: ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv', 'train_clean.csv', 'val_clean.csv', 'test_clean.csv', 'predictions_exp01_mt5small_zeroshot.csv']


In [4]:
# CELL S2 — Install all dependencies
!pip install -q --upgrade pip

!pip install -q \
    transformers==4.44.0 \
    datasets==2.19.0 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    bitsandbytes==0.43.1 \
    rouge-score==0.1.2 \
    sentencepiece==0.2.0 \
    protobuf==3.20.3 \
    evaluate==0.4.1 \
    fsspec==2025.3.0

# Verify no critical conflicts remain
import pkg_resources
packages = ['transformers', 'datasets', 'peft', 'accelerate', 'evaluate']
print("\n=== Installed versions ===")
for p in packages:
    try:
        print(f"  {p}: {pkg_resources.get_distribution(p).version}")
    except:
        print(f"  {p}: NOT FOUND")

print("\nInstallation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:00
ERROR: Cannot install datasets==2.19.0 and fsspec==2025.3.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts

=== Installed versions ===
  transformers: 4.40.0
  datasets: 2.19.0
  peft: 0.10.0
  accelerate: 0.29.3
  evaluate: 0.4.6

Installation complete.


/tmp/ipykernel_973/165690623.py:7: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [5]:

import importlib

required = {
    'transformers': '4.40.0',
    'datasets':     '2.19.0',
    'peft':         '0.10.0',
    'accelerate':   '0.29.3',
    'evaluate':     '0.4.6',
    'rouge_score':  None,
    'sentencepiece':None,
}

print("=== Package check ===")
all_ok = True
for package, expected in required.items():
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'installed')
        print(f"  ✓ {package}: {version}")
    except ImportError:
        print(f"  ✗ {package}: MISSING")
        all_ok = False

if all_ok:
    print("\nAll packages present. Proceed to Cell S3.")
else:
    print("\nSome packages missing — run: !pip install -q peft evaluate rouge-score sentencepiece")

=== Package check ===
  ✓ transformers: 4.40.0
  ✓ datasets: 2.19.0
  ✓ peft: 0.10.0
  ✓ accelerate: 0.29.3
  ✓ evaluate: 0.4.6
  ✓ rouge_score: installed
  ✓ sentencepiece: 0.2.1

All packages present. Proceed to Cell S3.


In [6]:
# imports
import gc
import os
import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import evaluate as hf_evaluate
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4


In [8]:
# CELL S4 — Load cleaned data
train = pd.read_csv(f'{DATA_DIR}/train_clean.csv')
val   = pd.read_csv(f'{DATA_DIR}/val_clean.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_clean.csv')

# Column mapping for this dataset
q_col    = 'input'
a_col    = 'output'
lang_col = 'subset'
id_col   = 'ID'

# Rebuild prompts v1 if missing
if 'input_text' not in train.columns:
    def _prompt_v1(q, l):
        return f"Language: {l}\nQuestion: {q}\nAnswer:"
    train['input_text']  = train.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)
    train['target_text'] = train[a_col].astype(str)
    val['input_text']    = val.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)
    val['target_text']   = val[a_col].astype(str)
    test['input_text']   = test.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print("Sample input:", train['input_text'].iloc[0])
print("Sample target:", train['target_text'].iloc[0][:150])

Train: 29814, Val: 6686, Test: 2618
Sample input: Language: Aka_Gha
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:
Sample target: Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na wɔagye wɔn nkate atom a wɔremmu atɛ


In [19]:
# CELL S5 — Load sample submission and verify format
sample = pd.read_csv(f'{DATA_DIR}/SampleSubmission.csv')
print(" Sample submission loaded")
print(f"  Columns : {sample.columns.tolist()}")
print(f"  Shape   : {sample.shape}")
print(f"\nFirst 2 rows:")
print(sample.head(2).to_string())

 Sample submission loaded
  Columns : ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
  Shape   : (2618, 4)

First 2 rows:
                       ID             TargetRLF1             TargetR1F1              TargetLLM
0  ID_TS_Aka_Gha_A3B1799D  Wuna dey craze, eweeh  Wuna dey craze, eweeh  Wuna dey craze, eweeh
1  ID_TS_Aka_Gha_1C80317F  Wuna dey craze, eweeh  Wuna dey craze, eweeh  Wuna dey craze, eweeh


In [20]:
# CELL S6 — All shared helper functions

#  1. ROUGE metric
rouge_metric = hf_evaluate.load("rouge")

#  2. Tokenize datasets
def get_tokenized_datasets(tokenizer, train_df, val_df,
                            max_input=256, max_target=256):
    def preprocess(examples):
        inputs = tokenizer(
            examples['input_text'],
            max_length=max_input,
            truncation=True,
            padding='max_length'
        )
        targets = tokenizer(
            examples['target_text'],
            max_length=max_target,
            truncation=True,
            padding='max_length'
        )
        labels = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in targets['input_ids']
        ]
        inputs['labels'] = labels
        return inputs

    train_ds  = Dataset.from_pandas(
        train_df[['input_text','target_text']].reset_index(drop=True))
    val_ds    = Dataset.from_pandas(
        val_df[['input_text','target_text']].reset_index(drop=True))
    train_tok = train_ds.map(preprocess, batched=True,
                             remove_columns=['input_text','target_text'])
    val_tok   = val_ds.map(preprocess,   batched=True,
                           remove_columns=['input_text','target_text'])
    return train_tok, val_tok

# 3. Compute metrics
def make_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        labels        = np.where(labels != -100, labels,
                                 tokenizer.pad_token_id)
        decoded_preds  = tokenizer.batch_decode(
            preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(
            labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]
        result = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        return {
            "rouge1": round(result["rouge1"], 4),
            "rougeL": round(result["rougeL"], 4)
        }
    return compute_metrics

# 4. Training
def run_training(model, tokenizer, train_tok, val_tok,
                 experiment_name, learning_rate,
                 num_epochs, batch_size, max_target_len):
    args = Seq2SeqTrainingArguments(
        output_dir=f"{OUT_DIR}/checkpoints_{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        learning_rate=learning_rate,
        weight_decay=0.01,
        fp16=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        predict_with_generate=True,
        generation_max_length=max_target_len,
        logging_steps=50,
        report_to="none",
        push_to_hub=False
    )
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
    trainer  = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=make_compute_metrics(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    est = len(train_tok) * num_epochs / (batch_size * 4) / 60
    print(f" Estimated time : ~{est:.0f} minutes")
    print("Starting training...")
    trainer.train()
    return trainer

# 5. Save model
def save_model(model, tokenizer, experiment_name):
    path = f'{OUT_DIR}/model_{experiment_name}'
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f" Model saved : {path}")
    return path

# 6. Generate answers
def generate_answers(model, tokenizer, texts,
                     max_input=256, max_target=256,
                     batch_size=16, num_beams=4):
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    all_answers = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt',
            max_length=max_input,
            truncation=True, padding=True
        )
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_target,
                num_beams=num_beams,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=1.0
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_answers.extend(decoded)
    return all_answers

# 7. Save predictions
def save_predictions(test_df, answers, experiment_name):
    out              = test_df.copy()
    out['predicted_answer'] = answers
    path             = f'{DATA_DIR}/predictions_{experiment_name}.csv'
    out.to_csv(path, index=False)
    print(f" Predictions saved : {path}")
    return path

# 8. Build Zindi submission
def build_submission(experiment_name):
    """
    Reads predictions CSV and builds correctly formatted
    Zindi submission with columns:
    ID | TargetRLF1 | TargetR1F1 | TargetLLM
    """
    pred_path = f'{DATA_DIR}/predictions_{experiment_name}.csv'
    if not os.path.exists(pred_path):
        print(f"✗ No predictions found for {experiment_name}")
        print(f"  Expected at: {pred_path}")
        return None

    preds       = pd.read_csv(pred_path)
    pred_id_col = preds.columns[0]
    pred_answer = preds['predicted_answer'].astype(str).values

    submission  = pd.DataFrame({
        'ID':         preds[pred_id_col].values,
        'TargetRLF1': pred_answer,
        'TargetR1F1': pred_answer,
        'TargetLLM':  pred_answer
    })

    # Validate
    assert list(submission.columns) == ['ID','TargetRLF1','TargetR1F1','TargetLLM'], \
        "Column mismatch!"
    assert len(submission) == len(sample), \
        f"Row mismatch: {len(submission)} vs {len(sample)}"
    assert submission.isnull().sum().sum() == 0, \
        "Null values found!"

    path = f'{SUB_DIR}/submission_{experiment_name}.csv'
    submission.to_csv(path, index=False)
    print(f" Submission saved  : {path}")
    print(f"  Shape             : {submission.shape}")
    print(f"  Columns           : {submission.columns.tolist()}")
    print(f"\nFirst 3 rows:")
    print(submission.head(3).to_string())
    return path

# 9. Plot learning curves
def plot_learning_curves(trainer, experiment_name):
    log      = trainer.state.log_history
    t_steps  = [x['step']      for x in log
                if 'loss' in x and 'eval_loss' not in x]
    t_losses = [x['loss']      for x in log
                if 'loss' in x and 'eval_loss' not in x]
    e_epochs = [x['epoch']     for x in log if 'eval_loss' in x]
    e_losses = [x['eval_loss'] for x in log if 'eval_loss' in x]
    e_r1     = [x.get('eval_rouge1', 0) for x in log if 'eval_loss' in x]
    e_rL     = [x.get('eval_rougeL', 0) for x in log if 'eval_loss' in x]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Learning Curves — {experiment_name}',
                 fontsize=13, fontweight='bold')

    axes[0].plot(t_steps, t_losses, color='royalblue', linewidth=1.5)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(e_epochs, e_losses, color='darkorange',
                 marker='o', linewidth=2)
    axes[1].set_title('Validation Loss per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Eval Loss')
    axes[1].grid(True, alpha=0.4)

    axes[2].plot(e_epochs, e_r1, color='green',
                 marker='o', linewidth=2, label='ROUGE-1')
    axes[2].plot(e_epochs, e_rL, color='purple',
                 marker='s', linewidth=2, label='ROUGE-L')
    axes[2].set_title('ROUGE Scores per Epoch')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.4)

    plt.tight_layout()
    path = f'{PLOT_DIR}/learning_curves_{experiment_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Learning curves saved : {path}")

#  10. Experiment log
LOG_PATH = f'{DATA_DIR}/experiment_log.json'

def save_experiment_log(experiment_name, config, metrics):
    log = {}
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, 'r') as f:
            log = json.load(f)
    log[experiment_name] = {"config": config, "metrics": metrics}
    with open(LOG_PATH, 'w') as f:
        json.dump(log, f, indent=2)
    print(f"✓ Experiment log updated : {LOG_PATH}")
    print(json.dumps(log[experiment_name], indent=2))

def update_zindi_score(experiment_name, score):
    if not os.path.exists(LOG_PATH):
        print("No log file found. Run the experiment first.")
        return
    with open(LOG_PATH, 'r') as f:
        log = json.load(f)
    if experiment_name not in log:
        print(f"Experiment {experiment_name} not in log.")
        return
    log[experiment_name]['metrics']['zindi_score'] = score
    with open(LOG_PATH, 'w') as f:
        json.dump(log, f, indent=2)
    print(f"✓ Zindi score updated: {experiment_name} = {score}")

# 11. Check if already done
def is_done(experiment_name):
    pred_path = f'{DATA_DIR}/predictions_{experiment_name}.csv'
    if os.path.exists(pred_path):
        print(f"✓ ALREADY DONE: {experiment_name}")
        print(f"  Predictions at: {pred_path}")
        print("  Skipping to submission cell.")
        return True
    print(f" NOT YET DONE: {experiment_name} — will run now.")
    return False

# 12. Free GPU memory
def free_gpu(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f" GPU memory freed")
        print(f"  Used : {used:.2f} GB")
        print(f"  Free : {free:.2f} GB")

print(" All helper functions loaded and ready.")

 All helper functions loaded and ready.


In [21]:
# CELL S7 — Check what experiments are already done
print("=== EXPERIMENT STATUS ===\n")

experiments_list = [
    "exp01_mt5small_zeroshot",
    "exp02_mt5small_lora_r16",
    "exp03_flanT5base_lora_r16",
    "exp04_flanT5base_lora_r32",
    "exp05_flanT5base_promptv2",
    "exp06_flanT5large_lora_r16"
]

for exp in experiments_list:
    pred_path = f'{DATA_DIR}/predictions_{exp}.csv'
    sub_path  = f'{SUB_DIR}/submission_{exp}.csv'
    done      = " DONE" if os.path.exists(pred_path) else " NOT DONE"
    submitted = " SUBMITTED" if os.path.exists(sub_path) else " NOT SUBMITTED"
    print(f"{exp}")
    print(f"  Predictions : {done}")
    print(f"  Submission  : {submitted}")

if os.path.exists(LOG_PATH):
    print("\n=== SCORES SO FAR ===")
    with open(LOG_PATH, 'r') as f:
        log = json.load(f)
    for name, entry in log.items():
        m = entry.get('metrics', {})
        print(f"  {name}")
        print(f"    ROUGE-1    : {m.get('val_rouge1', 'N/A')}")
        print(f"    ROUGE-L    : {m.get('val_rougeL', 'N/A')}")
        print(f"    Zindi Score: {m.get('zindi_score', 'not yet')}")
else:
    print("\nNo experiment log yet — starting fresh.")

=== EXPERIMENT STATUS ===

exp01_mt5small_zeroshot
  Predictions :  DONE
  Submission  :  NOT SUBMITTED
exp02_mt5small_lora_r16
  Predictions :  NOT DONE
  Submission  :  NOT SUBMITTED
exp03_flanT5base_lora_r16
  Predictions :  NOT DONE
  Submission  :  NOT SUBMITTED
exp04_flanT5base_lora_r32
  Predictions :  NOT DONE
  Submission  :  NOT SUBMITTED
exp05_flanT5base_promptv2
  Predictions :  NOT DONE
  Submission  :  NOT SUBMITTED
exp06_flanT5large_lora_r16
  Predictions :  NOT DONE
  Submission  :  NOT SUBMITTED

No experiment log yet — starting fresh.


In [22]:
# EXPERIMENT 1 — mt5-small Zero-Shot (No Training)
# WHAT    : Run mt5-small on test set without any fine-tuning
# WHY     : Establishes a baseline. Shows how badly the model
#           performs without task-specific training. All other
#           experiments are compared against this score.
# CHANGE  : Nothing — pure zero-shot inference

EXP1 = "exp01_mt5small_zeroshot"

if not is_done(EXP1):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT 1 : {EXP1}")
    print(f"Model        : google/mt5-small")
    print(f"Training     : NONE (zero-shot)")
    print(f"{'='*60}\n")

    # Load model
    tok1   = AutoTokenizer.from_pretrained("google/mt5-small")
    model1 = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
    print(f"Parameters : {model1.num_parameters():,}")

    # Zero-shot inference on test set
    print("\nRunning zero-shot inference on test set...")
    answers1 = generate_answers(
        model1, tok1,
        test['input_text'].tolist(),
        max_input=256, max_target=256,
        batch_size=16, num_beams=4
    )
    save_predictions(test, answers1, EXP1)

    # Evaluate on validation set for local ROUGE score
    print("\nEvaluating on validation set...")
    val_ans1 = generate_answers(
        model1, tok1,
        val['input_text'].tolist(),
        max_input=256, max_target=256,
        batch_size=16, num_beams=4
    )
    r1 = rouge_metric.compute(
        predictions=[a.strip() for a in val_ans1],
        references=[str(t).strip() for t in val['target_text'].tolist()],
        use_stemmer=True
    )
    print(f"\n✓ Val ROUGE-1 : {r1['rouge1']:.4f}")
    print(f"✓ Val ROUGE-L : {r1['rougeL']:.4f}")

    # Save to experiment log
    save_experiment_log(
        EXP1,
        config={
            "model": "google/mt5-small",
            "lora": False,
            "training": False,
            "prompt_version": "v1",
            "purpose": "Zero-shot baseline"
        },
        metrics={
            "val_rouge1": round(r1['rouge1'], 4),
            "val_rougeL": round(r1['rougeL'], 4),
            "zindi_score": "pending"
        }
    )

    # Free GPU memory before next experiment
    free_gpu(model1, tok1)
    print(f"\n{'='*60}")
    print("EXPERIMENT 1 COMPLETE")
    print(f"{'='*60}")

✓ ALREADY DONE: exp01_mt5small_zeroshot
  Predictions at: /content/drive/MyDrive/multilingual_health_qa/data/predictions_exp01_mt5small_zeroshot.csv
  Skipping to submission cell.


In [25]:
# EXPERIMENT 1 — Build submission and download
sub_path1 = build_submission(EXP1)
files.download(sub_path1)
print("\n NEXT STEPS:")
print("1. Go to Zindi competition → Submit tab")
print("2. Upload the downloaded file")
print("3. Comment: 'Exp 1: mt5-small zero-shot baseline'")
print("4. Wait for score → screenshot it")
print("5. Run the cell below with your real score")

 Submission saved  : /content/drive/MyDrive/multilingual_health_qa/submissions/submission_exp01_mt5small_zeroshot.csv
  Shape             : (2618, 4)
  Columns           : ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']

First 3 rows:
                       ID       TargetRLF1       TargetR1F1        TargetLLM
0  ID_TS_Aka_Gha_A3B1799D    <extra_id_0>.    <extra_id_0>.    <extra_id_0>.
1  ID_TS_Aka_Gha_1C80317F  <extra_id_0>?e蝾  <extra_id_0>?e蝾  <extra_id_0>?e蝾
2  ID_TS_Aka_Gha_06671AD1    <extra_id_0>.    <extra_id_0>.    <extra_id_0>.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 NEXT STEPS:
1. Go to Zindi competition → Submit tab
2. Upload the downloaded file
3. Comment: 'Exp 1: mt5-small zero-shot baseline'
4. Wait for score → screenshot it
5. Run the cell below with your real score


In [26]:
# EXPERIMENT 1 — Zindi score
ZINDI_SCORE_EXP1 = 0.00167

update_zindi_score(EXP1, ZINDI_SCORE_EXP1)
print(f"\nExp 1 score recorded: {ZINDI_SCORE_EXP1}")
print("Now commit to GitHub, then scroll down to Experiment 2.")

No log file found. Run the experiment first.

Exp 1 score recorded: 0.00167
Now commit to GitHub, then scroll down to Experiment 2.
